In [1]:
# connect to Gemini API
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

# 1. Initialize the Gemini model
# Note: It automatically picks up the GOOGLE_API_KEY from environment variables.
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash", # this model works
    temperature=0.7,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)



Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


### Open Information Extraction 

Example prompt from the paper


>Given a piece of text, extract relational triplets in
the form of [Subject, Relation, Object] from it.
Here are some examples:
Example 1:
Text: The 17068.8 millimeter long ALCO RS-3
has a diesel-electric transmission.
Triplets: [[‘ALCO RS-3’, ‘powerType’, ‘Dieselelectric transmission’], [‘ALCO RS-3’, ‘length’,
‘17068.8 (millimetres)’]] ...
Now please extract triplets from the following
text: Alan Shepard was born on Nov 18, 1923
and selected by NASA in 1959. He was a member of the Apollo 14 crew.


I will try to replicate this behaviour now

In [2]:
systemPrompt = """You are an expert in ontology, semantic relations and ontology."""
userPrompt = """
                Given a piece of text, extract relational triplets in
                the form of [Subject, Relation, Object] from it.
                Here are some examples:
                Example 1:
                Text: The 17068.8 millimeter long ALCO RS-3
                has a diesel-electric transmission.
                Triplets: [[‘ALCO RS-3’, ‘powerType’, ‘Dieselelectric transmission’], [‘ALCO RS-3’, ‘length’,
                ‘17068.8 (millimetres)’]] ...
                Now please extract triplets from the following
                text: Alan Shepard was born on Nov 18, 1923
                and selected by NASA in 1959. He was a member of the Apollo 14 crew
            """

In [14]:
messages = [
    SystemMessage(content=systemPrompt),
    HumanMessage(content=userPrompt)
]

try:

    response = llm.invoke(messages)
    print("Response Generated")
    print(response.content)

except Exception as e:

    print(f"Failed to generate a response with error: {e}")


/home/anubhabgiri/Documents/projects/knowledge-graph/.venv/lib64/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Response Generated
[{'type': 'text', 'text': 'Based on the text provided, here are the extracted relational triplets:\n\nTriplets:\n[\n  [‘Alan Shepard’, ‘birthDate’, ‘1923-11-18’],\n  [‘Alan Shepard’, ‘agency’, ‘NASA’],\n  [‘Alan Shepard’, ‘selectedByNASA’, ‘1959’],\n  [‘Alan Shepard’, ‘mission’, ‘Apollo 14’]\n]', 'extras': {'signature': 'Et4bCtsbAWkUfRN/Y1l4YhCUwbg69UQt/ozUIa7tJebjaKSOlqnG47CnKbwgkwLlfNvToAjCVKN8gz1Z7VvM3AmpmxyA+njy7cwE+zT0jSuxTZygHe6srSam046YOLspulc09s+ogigz/JafcYls38VpYyFSs26jh1ewucCjLmdM0ryPZbZA5hF7sFuq+Ls04XWPMsuBmOadcvoMe8BWXT3jIR5trk0zoMuOOXA7jeLDUQtVCBJYiUv8if3JOBEB7OOLo9c4i/2DZ6TYAehDiPf6L4GVl+aM2ZN5gRicYAjBspQLSg4S/ynz8ce3PMpEbJeHam46byTuuxkXWhAuS3LuUT+7zVuOYb0DYfRkf+B7cSY4P4Dm62dpC/0rrNor6w6owksbI+FXlKLwc2SMVve3fFBvXO9WS8adyG4I7V0F1pIBvmqcYwjyiY5yT9rnhXwObdobsxfJv//vSsIAdxqsm1Px8DnlKf0Dqt4n2qGS1++6iNQPeNkKWrXM3HPZbHIxzbHYPOQMjmIAKl5QIEDzrtRByKN01dCZmQenXLQR+u8nQi1If6sBzEeskAgO/edy0vGWHkw/wLyX0g4GvpktFFf2alNPpTvjxx6Qekl8YYHcH84a+/cLOrIFkUtKc/ltmHeZDQVPADxfSY

In [ ]:
# let's try to extract the response

response_text = response.content[0]["text"]
print(response_text)

Based on the text provided, here are the extracted relational triplets:

Triplets:
[
  [‘Alan Shepard’, ‘birthDate’, ‘1923-11-18’],
  [‘Alan Shepard’, ‘agency’, ‘NASA’],
  [‘Alan Shepard’, ‘selectedByNASA’, ‘1959’],
  [‘Alan Shepard’, ‘mission’, ‘Apollo 14’]
]


In [18]:
# storing the response in a more readable format

triplets = [
  ["Alan Shepard", "birthDate", "1923-11-18"],
  ["Alan Shepard", "agency", "NASA"],
  ["Alan Shepard", "selectedByNASA", "1959"],
  ["Alan Shepard", "mission", "Apollo 14"]
]

Successfully completed the first step of the framework!

### Schema Definition

sample prompt from the paper

> Given a piece of text and a list of relational triplets
extracted from it, write a definition for each relation present.
Example 1:
Text: The 17068.8 millimeter long ALCO RS-3
has a diesel-electric transmission.
Triplets: [[‘ALCO RS-3’, ‘powerType’, ‘Dieselelectric transmission’], [‘ALCO RS-3’, ‘length’,
‘17068.8 (millimetres)’]]
Definitions:
powerType: The subject entity uses the type of
power or energy source specified by the object
entity.
...
Now write a definition for each relation present
in the triplets extracted from the following text:
Text: Alan Shepard was an American who was
born on Nov 18, 1923 in New Hampshire, was
selected by NASA in 1959, was a member of the
Apollo 14 crew and died in California
Triplets: [[‘Alan Shepard’, ‘bornOn’, ‘Nov 18,
1923’], [‘Alan Shepard’, ‘participatedIn’, ‘Apollo14’]]

In [6]:
userPrompt = """
            Given a piece of text and a list of relational triplets
            extracted from it, write a definition for each relation present.
            Example 1:
            Text: The 17068.8 millimeter long ALCO RS-3
            has a diesel-electric transmission.
            Triplets: [[‘ALCO RS-3’, ‘powerType’, ‘Dieselelectric transmission’], [‘ALCO RS-3’, ‘length’,
            ‘17068.8 (millimetres)’]]
            Definitions:
            powerType: The subject entity uses the type of
            power or energy source specified by the object
            entity.
            ...
            Now write a definition for each relation present
            in the triplets extracted from the following text:
            Text: Alan Shepard was an American who was
            born on Nov 18, 1923 in New Hampshire, was
            selected by NASA in 1959, was a member of the
            Apollo 14 crew and died in California
            Triplets: [
                        [‘Alan Shepard’, ‘birthDate’, ‘1923-11-18’],
                        [‘Alan Shepard’, ‘agency’, ‘NASA’],
                        [‘Alan Shepard’, ‘selectedByNASA’, ‘1959’],
                        [‘Alan Shepard’, ‘mission’, ‘Apollo 14’]
                       ]
            """

In [7]:
messages = [
    SystemMessage(content=systemPrompt),
    HumanMessage(content=userPrompt)
]

try:

    response = llm.invoke(messages)
    print("Response Generated")
    print(response.content)

except Exception as e:

    print(f"Failed to generate a response with error: {e}")


/home/anubhabgiri/Documents/projects/knowledge-graph/.venv/lib64/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Response Generated
[{'type': 'text', 'text': 'Definitions:\n\n* **birthDate**: The exact calendar date on which the subject entity was born.\n* **agency**: The organization, institution, or governmental body with which the subject entity is affiliated or employed.\n* **selectedByNASA**: The year or date on which the subject entity was officially chosen or recruited by NASA to be part of a space program or astronaut group.\n* **mission**: The specific spaceflight, operational assignment, or task in which the subject entity participated as a crew member or participant.', 'extras': {'signature': 'EpUWCpIWAWkUfROE94OwDrX785OfxFEy0X4UfQiZAsMaOwwcydWXdsEs7SLn1DMMn7SyRAVmzzhsw1ceTaZs6CtNt7LS3OHLRUSjJNtLiCOsCeYDxCWDNnrAUSxcacr9AwpPDH5213Phe3GS+vU9ejmcR0ZwusSsQobRF882B6+qYGaNY0e/I18FgLn4S0nOmLx4K3zKCtOdq61sTFSzdX1pfY0sFHd80Qdl81PYDDRZoLuLdZozyg//L8yatSRmpwh+NFh7Rek77HgyhKTvT3mxOn+nh5Mkvqh57Ckug93VktYo3atQMef3m5IK8DKGT6tdRBd2uBkJKiXr/20qoZKAFTSiSGClu+vywbFREpazc+pWzVWedo6MX6SsmMd46Msw1ve7LJ+nBfd

In [8]:
response_text = response.content[0]["text"]
print(response_text)

Definitions:

* **birthDate**: The exact calendar date on which the subject entity was born.
* **agency**: The organization, institution, or governmental body with which the subject entity is affiliated or employed.
* **selectedByNASA**: The year or date on which the subject entity was officially chosen or recruited by NASA to be part of a space program or astronaut group.
* **mission**: The specific spaceflight, operational assignment, or task in which the subject entity participated as a crew member or participant.


Successfully completed the second step! 

sample prompt from the paper

> Given a piece of text, a relational triplet extracted
from it, and the definition of the relation in it,
choose the most appropriate relation to replace it
in this context if there is any.
Text: Alan Shepard was born on Nov 18, 1923
and selected by NASA in 1959. He was a member
of the Apollo 14 crew.
Triplets: [‘Alan Shepard’, ‘participatedIn’,
‘Apollo 14’]
Definition of ‘participatedIn’: The subject entity
took part in the event or mission specified by the
object entity.
Choices:
A. ‘mission’: The subject entity participated in
the event or operation specified by the object entity.
B. ‘season’: The subject entity participated in the
season of a series specified by the object entity.
...
F. None of the above

In [12]:
userPrompt = """
Given a piece of text, a relational triplet extracted
from it, and the definition of the relation in it,
choose the most appropriate relation to replace it
in this context if there is any.
Text: Alan Shepard was born on Nov 18, 1923
and selected by NASA in 1959. He was a member
of the Apollo 14 crew.
Triplets: [‘Alan Shepard’, ‘mission’, ‘Apollo 14’]

Definitions:

* **birthDate**: The exact calendar date on which the subject entity was born.
* **agency**: The organization, institution, or governmental body with which the subject entity is affiliated or employed.
* **selectedByNASA**: The year or date on which the subject entity was officially chosen or recruited by NASA to be part of a space program or astronaut group.
* **mission**: The specific spaceflight, operational assignment, or task in which the subject entity participated as a crew member or participant.

Choices:
A. ‘mission’: The subject entity participated in
the event or operation specified by the object entity.
B. ‘season’: The subject entity participated in the
season of a series specified by the object entity.
...
F. None of the above
"""

In [13]:
messages = [
    SystemMessage(content=systemPrompt),
    HumanMessage(content=userPrompt)
]

try:

    response = llm.invoke(messages)
    print("Response Generated")
    print(response.content)

except Exception as e:

    print(f"Failed to generate a response with error: {e}")

/home/anubhabgiri/Documents/projects/knowledge-graph/.venv/lib64/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Response Generated
[{'type': 'text', 'text': 'Based on the provided text, triplet, and definitions, the relation **‘mission’** accurately captures the relationship between Alan Shepard and Apollo 14.\n\n**Correct Answer:** **A. ‘mission’: The subject entity participated in the event or operation specified by the object entity.**', 'extras': {'signature': 'EqENCp4NAWkUfRM9oXtHlkiXamjLaS9PB58mlWoewJZTCQ+51bkHYrnszz+HE5Y1SZ4S8dEG6MyUE8JoaqgmxNwg97Lk/zZGidD3QFOaMpIk/8hqySU7ggyAYIRotW0hwurOttCuvYU/GGoOvYtFKL+iskXX5/vL4N2VABzUzo+DL8BPq0lwtSze+JR2BMVjkMw9JkJBIY2+3PygJCl6CLWn5FQgX9t1z32EIhRl+9QVPt6lesS6p7IN3mZYhg4df5wrTAqAWPTH9ywqJJoWPCkr5+C6RPdGN9TB98iCWEFuodTJNk4CM3ROHJk0oYUenQA49w0+fEMy8/Dt5JwxTc0fDbA+fwFQq5KqVjOSXcvXJyfMbgORbq4eV8386P0ANsLWsFKnW63WXJAuLR268PYAY2M86hlNeH7BxDqRvBcEgZq3/dnNa25TbnqXJ4qElp76b5bjdC6aLm0cHsu4AxPenliiuy03HVd3w0xLynLd2LymjogfT0WwAzie7F8rsLOn/P5JIE5yhUOhV4UolcPFC8L5uFYX3eAag47gbIPyD2DuVmBkzm47SuRwy68qrId9FHh1fClGVHvB+m+8O7I1bfDgdwJxBAdxSTrPh+bJmOfNp9aNdHV2WxbAkhlxj8

In [14]:
response_text = response.content[0]["text"]
print(response_text)

Based on the provided text, triplet, and definitions, the relation **‘mission’** accurately captures the relationship between Alan Shepard and Apollo 14.

**Correct Answer:** **A. ‘mission’: The subject entity participated in the event or operation specified by the object entity.**


Successfully completed the third step!

Next step is to collate all the triplets together after canonicalization.  
*Open question*: How do we generate the choices used here? 

In the paper, it is mentioned that the choices above are obtained by using vector similarity search. 

In [1]:
# A sample target schema dictionary (Canonical Relation -> Definition)
# In real-world use, this can be loaded from a CSV, JSON, or ontology file (e.g. Wikidata / WebNLG)
# In the previous step I had provided hard coded choices
# Now I will attempt to generate them dynamically using vector similarity
TARGET_SCHEMA = {
    "mission": "The subject entity participated in the event or operation specified by the object entity.",
    "season": "The subject entity participated in the season of a series specified by the object entity.",
    "league": "The subject entity played in or competed in the sports league specified by the object entity.",
    "almaMater": "The educational institution or university attended by the subject entity.",
    "agency": "The organization, institution, or governmental body with which the subject entity is affiliated or employed.",
    "birthDate": "The exact calendar date on which the subject entity was born.",
    "deathPlace": "The geographical location where the subject entity passed away.",
    "award": "The award, medal, or recognition received by the subject entity.",
    "draftPick": "The selection round or pick by which a sports team drafted the subject entity.",
    "occupation": "The profession, career, or primary work role of the subject entity."
}

In [2]:
import math
from sentence_transformers import SentenceTransformer

# Load a lightweight local embedding model (runs fast on CPU)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# 1. Pre-compute and cache embeddings for target schema definitions
schema_items = list(TARGET_SCHEMA.items())  # [(rel, def), ...]
schema_texts = [f"{rel}: {defn}" for rel, defn in schema_items]
schema_embeddings = embedder.encode(schema_texts, normalize_embeddings=True)

def cosine_similarity(vec1, vec2):
    return sum(a * b for a, b in zip(vec1, vec2))

def retrieve_top_k_choices(query_relation: str, query_definition: str, top_k: int = 5):
    """
    Computes cosine similarity between the extracted relation's definition
    and all schema definitions, returning the top-k choices formatted as MCQ.
    """
    query_text = f"{query_relation}: {query_definition}"
    query_emb = embedder.encode(query_text, normalize_embeddings=True)
    
    # Compute similarity against all schema elements
    scores = []
    for idx, (rel, defn) in enumerate(schema_items):
        sim = cosine_similarity(query_emb, schema_embeddings[idx])
        scores.append((sim, rel, defn))
    
    # Sort descending by similarity score
    scores.sort(key=lambda x: x[0], reverse=True)
    top_candidates = scores[:top_k]
    
    # Format as Multiple-Choice Options (A, B, C, ...)
    letters = ["A", "B", "C", "D", "E", "F", "G"]
    choices_str = ""
    candidate_map = {}
    
    for i, (_, rel, defn) in enumerate(top_candidates):
        letter = letters[i]
        choices_str += f"{letter}. '{rel}': {defn}\n"
        candidate_map[letter] = rel
        
    # Always append 'None of the above' as the last choice
    none_letter = letters[len(top_candidates)]
    choices_str += f"{none_letter}. None of the above\n"
    candidate_map[none_letter] = "None of the above"
    
    return choices_str, candidate_map

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
relationName = "participatedIn"
relationDefinition = "The subject entity took part in the event or mission specified by the object entity."

choices_text, candidate_map = retrieve_top_k_choices(relationName, relationDefinition, top_k=5)
print(choices_text)
print(candidate_map)

A. 'mission': The subject entity participated in the event or operation specified by the object entity.
B. 'season': The subject entity participated in the season of a series specified by the object entity.
C. 'agency': The organization, institution, or governmental body with which the subject entity is affiliated or employed.
D. 'league': The subject entity played in or competed in the sports league specified by the object entity.
E. 'award': The award, medal, or recognition received by the subject entity.
F. None of the above

{'A': 'mission', 'B': 'season', 'C': 'agency', 'D': 'league', 'E': 'award', 'F': 'None of the above'}


This is how we can generate top N choices dynamically and send it along with the step 3 LLM call and canonicalize the relations